In [1]:
# ================================================================
#  LCCN-R22c - MISSING TABLE 8 SEEDS: XLM-RoBERTa  [GPU, ~6 h]
#  Answers IJIES Reviewer 1, point 3. The trainer kept only the last
#  seed's predictions per arm, so these per-seed runs were never stored:
#    seed 43: raw, indicsoundex, lccn
#    seeds 43 and 44: lccn_base, lccn_indicxlit
#  Seven XLM-RoBERTa runs (trainer v1.4, BUDGET unchanged) on the NB1
#  arm texts, each written as preds/XLM-RoBERTa__{arm}__seed{N}.csv.
#  Afterwards re-run lccn-r13-perseed-stats with THIS output attached.
#  Add Input: paper1-nb1-lccn-final (output: splits/),
#             paper1-nb2-lccn-ml (output: trainer.py).
#  Accelerator GPU T4 x2 or P100. Internet ON. Save & Run All (Commit).
# ================================================================
import subprocess, sys, os, shutil, glob
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode
sh('pip install -q scikit-learn xgboost scipy')
sh('pip install -q transformers accelerate --upgrade')
for p in ("trainer.py", "/kaggle/working/trainer.py"):
    if os.path.exists(p): os.remove(p)
shutil.rmtree("__pycache__", ignore_errors=True); sys.modules.pop("trainer", None)
tc = [p for p in sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True)) if "nb2" in p]
tc = tc or sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True))
assert tc, "trainer.py not found - attach the paper1-nb2-lccn-ml output"
shutil.copy(tc[0], "/kaggle/working/trainer.py")
import torch, transformers
print("\npython", sys.version.split()[0], "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU is off - Settings > Accelerator > GPU"
print("gpu:", torch.cuda.get_device_name(0))


+ pip install -q scikit-learn xgboost scipy
+ pip install -q transformers accelerate --upgrade
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 17.2 MB/s eta 0:00:00

python 3.13.15 | torch 2.11.0+cu128 | transformers 5.18.0 | cuda True
gpu: Tesla T4


In [2]:
# ================================================================
#  CELL 1 - SETUP (trainer v1.4 asserted; NB1 splits carry all arms)
# ================================================================
import importlib, trainer
importlib.reload(trainer)
from trainer import *
import pandas as pd, numpy as np
FP = trainer.fingerprint(); assert trainer.TRAINER_VERSION == "1.4" and FP == "5b607248090bd26e", FP
sd = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)]
assert sd, "attach the paper1-nb1-lccn-final output"
tr, va, te, HAVE = load_splits(sd[0])
for a in ("raw", "indicsoundex", "lccn", "lccn_base", "lccn_indicxlit"):
    assert f"text_{a}" in tr.columns, a
print("arms available:", HAVE)


trainer v1.4  sha256:5b607248090bd26e
  BUDGET {'epochs': 10, 'lr': 2e-05, 'batch_size': 32, 'max_len': 96, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'patience': 3}
  DEEP   {'epochs': 8, 'lr': 0.001, 'batch_size': 64, 'max_len': 64, 'vocab': 40000, 'emb': 200, 'hid': 128, 'dropout': 0.3, 'patience': 2}
test-set id hash: e2d6e1f7d564cace
train  28,000  {'disgust': np.int64(7000), 'happy': np.int64(7000), 'sad': np.int64(7000), 'anger': np.int64(7000)}
val     6,000  {'anger': np.int64(1500), 'happy': np.int64(1500), 'sad': np.int64(1500), 'disgust': np.int64(1500)}
test    6,000  {'disgust': np.int64(1500), 'anger': np.int64(1500), 'sad': np.int64(1500), 'happy': np.int64(1500)}
arms found: ['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn', 'lccn_itrans', 'lccn_indicxlit']
arms available: ['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn

In [3]:
# ================================================================
#  CELL 2 - THE SEVEN PER-SEED RUNS (resumable: a __seed file that
#  already exists in this session's output or an attached earlier
#  attempt is not re-run)
# ================================================================
MODEL = "XLM-RoBERTa"
RUNS = [(43, "raw"), (43, "indicsoundex"), (43, "lccn"),
        (43, "lccn_base"), (44, "lccn_base"), (43, "lccn_indicxlit"), (44, "lccn_indicxlit")]
have = {os.path.basename(f) for f in glob.glob("/kaggle/input/**/preds/*__seed*.csv", recursive=True)
        + glob.glob("/kaggle/working/preds/*__seed*.csv")}
rows = []
for seed, arm in RUNS:
    tag_file = f"{MODEL}__{arm}__seed{seed}.csv"
    if tag_file in have:
        print(f"skip {arm} seed {seed}: {tag_file} already exists"); continue
    p = f"/kaggle/working/preds/{MODEL}__{arm}.csv"
    if os.path.exists(p): os.remove(p)
    r = run_grid(tr, va, te, [arm], [MODEL], tag=f"lccn_r22c_{arm}_seed{seed}", seeds_tf=(seed,), resume=False)
    shutil.copy(p, f"/kaggle/working/preds/{tag_file}")
    f1 = float(r.iloc[0][arm]); rows.append(dict(arm=arm, seed=seed, macro_f1=round(f1, 4)))
    print(f"  {arm:<16} seed {seed}: macro-F1 {f1:.4f}  -> preds/{tag_file}")
pd.DataFrame(rows).to_csv("/kaggle/working/results_lccn_r22c_seeds.csv", index=False)
print("\n", pd.DataFrame(rows).to_string(index=False))
print("\nStored seed references: raw 42 0.8058 / 44 0.8032; indicsoundex 42 0.7773 / 44 0.7839;")
print("lccn 42 0.7991 / 44 0.8066; lccn_base 42 0.7991; lccn_indicxlit 42 0.8130.")
print("DONE - attach this output to lccn-r13-perseed-stats and re-run it unchanged.")



=== XLM-RoBERTa / raw ===


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  1/10  val macro-F1 = 0.7079
    epoch  2/10  val macro-F1 = 0.8020
    epoch  3/10  val macro-F1 = 0.8068
    epoch  4/10  val macro-F1 = 0.8149
    epoch  5/10  val macro-F1 = 0.8140
    epoch  6/10  val macro-F1 = 0.8177
    epoch  7/10  val macro-F1 = 0.8154
    epoch  8/10  val macro-F1 = 0.8213
    epoch  9/10  val macro-F1 = 0.8192
    epoch 10/10  val macro-F1 = 0.8163
  -> macro-F1 0.8146

wrote /kaggle/working/results_lccn_r22c_lccn_indicxlit_seed43.csv
  lccn_indicxlit   seed 43: macro-F1 0.8146  -> preds/XLM-RoBERTa__lccn_indicxlit__seed43.csv

=== XLM-RoBERTa / lccn_indicxlit ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  1/10  val macro-F1 = 0.7617
    epoch  2/10  val macro-F1 = 0.7965
    epoch  3/10  val macro-F1 = 0.8070
    epoch  4/10  val macro-F1 = 0.8183
    epoch  5/10  val macro-F1 = 0.8183
    epoch  6/10  val macro-F1 = 0.8256
    epoch  7/10  val macro-F1 = 0.8198
    epoch  8/10  val macro-F1 = 0.8254
    epoch  9/10  val macro-F1 = 0.8232
    early stop at epoch 9
  -> macro-F1 0.8196

wrote /kaggle/working/results_lccn_r22c_lccn_indicxlit_seed44.csv
  lccn_indicxlit   seed 44: macro-F1 0.8196  -> preds/XLM-RoBERTa__lccn_indicxlit__seed44.csv

            arm  seed  macro_f1
           raw    43    0.8067
  indicsoundex    43    0.7797
          lccn    43    0.8069
     lccn_base    43    0.8062
     lccn_base    44    0.8057
lccn_indicxlit    43    0.8146
lccn_indicxlit    44    0.8196

Stored seed references: raw 42 0.8058 / 44 0.8032; indicsoundex 42 0.7773 / 44 0.7839;
lccn 42 0.7991 / 44 0.8066; lccn_base 42 0.7991; lccn_indicxlit 42 0.8130.
DONE - attach this output to